# Better instruction-following: GPT-2 large + Stanford Alpaca + LoRA (from scratch)
This notebook improves on the first instruction-tuned model (GPT-2 medium, ~935 examples) in three ways:

1. **Bigger base model:** GPT-2 large (774M parameters) instead of GPT-2 medium (355M).
2. **About 50x more instruction data:** Alpaca-cleaned (~52,000 examples, the cleaned version of the Stanford Alpaca dataset) instead of ~935 examples.
3. **LoRA, implemented from scratch:** the 774M base weights stay frozen and only small low-rank matrices are trained, so it fits on one free T4 GPU.

It also uses mixed precision (fp16) and gradient checkpointing to fit in GPU memory, and masks the prompt tokens in the loss, so the model learns to write answers rather than repeat instructions.

The model is evaluated on the **same 110 test instructions** as before, with the same Llama 3 judge, so the scores are directly comparable.

**Kaggle settings:** Accelerator → **GPU T4 x2**, Internet → **On**. Then Save Version → **Save & Run All (Commit)**. Expected time: about 2–4 hours.

In [ ]:
# ---- Setup ----
%pip install -q tiktoken safetensors huggingface_hub

import os, json, math, time, gc, random, urllib.request
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduces GPU memory fragmentation
import torch
import torch.nn as nn
import tiktoken
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
RESULTS = {}

def download(url, file_path):
    if not os.path.exists(file_path):
        urllib.request.urlretrieve(url, file_path)
    return file_path

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Working directory:", WORK_DIR)
print("Device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

# ---- Settings you can change ----
BASE_MODEL = "gpt2-large (774M)"   # or "gpt2-medium (355M)" for a faster run
NUM_TRAIN_EXAMPLES = None          # None = all ~51,000 Alpaca examples; e.g. 20000 for a shorter run
NUM_EPOCHS = 1
MAX_LENGTH = 400                   # examples longer than this (in tokens) are skipped (keeps GPU memory safe on a T4)
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4               # effective batch size = 4 x 4 = 16
LEARNING_RATE = 2e-4
LORA_RANK, LORA_ALPHA = 16, 32
MAX_TRAIN_MINUTES = 300            # training stops safely after this, so the run finishes within Kaggle's limits
MODEL_FILE = "gpt2-large774M-alpaca-lora.pth"

## 1. The GPT model (same architecture as in the project)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, _ = x.shape
        # [b, T, d_out] -> [b, num_heads, T, head_dim]
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)                      # [b, heads, T, T]
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)                   # causal mask before softmax
        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
                                      cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x                                   # Pre-LN: shortcut taken before LayerNorm
        x = self.drop_shortcut(self.att(self.norm1(x)))
        x = x + shortcut
        shortcut = x
        x = self.drop_shortcut(self.ff(self.norm2(x)))
        return x + shortcut


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)                        # [b, T, vocab_size]


def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]              # crop to the supported context length
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]         # only the last position
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < top_logits[:, -1:], torch.full_like(logits, -torch.inf), logits)
        if temperature > 0.0:
            probs = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and (idx_next == eos_id).all():
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

## 2. Loading OpenAI's pretrained GPT-2 weights into our model

In [ ]:
BASE_CONFIG = {"vocab_size": 50257, "context_length": 1024, "drop_rate": 0.0, "qkv_bias": True}
MODEL_CONFIGS = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12, "hf_repo": "openai-community/gpt2"},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16, "hf_repo": "openai-community/gpt2-medium"},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20, "hf_repo": "openai-community/gpt2-large"},
}

def get_config(model_name):
    cfg = dict(BASE_CONFIG)
    cfg.update({k: v for k, v in MODEL_CONFIGS[model_name].items() if k != "hf_repo"})
    return cfg

def download_gpt2_state_dict(model_name):
    repo = MODEL_CONFIGS[model_name]["hf_repo"]
    try:
        state_dict = load_file(hf_hub_download(repo_id=repo, filename="model.safetensors"))
    except Exception:
        state_dict = torch.load(hf_hub_download(repo_id=repo, filename="pytorch_model.bin"), map_location="cpu")
    return {(k[len("transformer."):] if k.startswith("transformer.") else k): v for k, v in state_dict.items()}

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {tuple(left.shape)}, Right: {tuple(right.shape)}")
    return nn.Parameter(right.clone().detach().to(left.dtype))

def load_weights_into_gpt(gpt, sd):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, sd["wpe.weight"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, sd["wte.weight"])
    d = gpt.tok_emb.weight.shape[1]
    for b, block in enumerate(gpt.trf_blocks):
        p = f"h.{b}."
        # OpenAI's Conv1D layers store weights as [in, out]; nn.Linear expects [out, in], hence .T
        q_w, k_w, v_w = torch.split(sd[p + "attn.c_attn.weight"], d, dim=-1)
        q_b, k_b, v_b = torch.split(sd[p + "attn.c_attn.bias"], d, dim=-1)
        block.att.W_query.weight = assign(block.att.W_query.weight, q_w.T)
        block.att.W_key.weight = assign(block.att.W_key.weight, k_w.T)
        block.att.W_value.weight = assign(block.att.W_value.weight, v_w.T)
        block.att.W_query.bias = assign(block.att.W_query.bias, q_b)
        block.att.W_key.bias = assign(block.att.W_key.bias, k_b)
        block.att.W_value.bias = assign(block.att.W_value.bias, v_b)
        block.att.out_proj.weight = assign(block.att.out_proj.weight, sd[p + "attn.c_proj.weight"].T)
        block.att.out_proj.bias = assign(block.att.out_proj.bias, sd[p + "attn.c_proj.bias"])
        block.ff.layers[0].weight = assign(block.ff.layers[0].weight, sd[p + "mlp.c_fc.weight"].T)
        block.ff.layers[0].bias = assign(block.ff.layers[0].bias, sd[p + "mlp.c_fc.bias"])
        block.ff.layers[2].weight = assign(block.ff.layers[2].weight, sd[p + "mlp.c_proj.weight"].T)
        block.ff.layers[2].bias = assign(block.ff.layers[2].bias, sd[p + "mlp.c_proj.bias"])
        block.norm1.scale = assign(block.norm1.scale, sd[p + "ln_1.weight"])
        block.norm1.shift = assign(block.norm1.shift, sd[p + "ln_1.bias"])
        block.norm2.scale = assign(block.norm2.scale, sd[p + "ln_2.weight"])
        block.norm2.shift = assign(block.norm2.shift, sd[p + "ln_2.bias"])
    gpt.final_norm.scale = assign(gpt.final_norm.scale, sd["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, sd["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, sd["wte.weight"])   # weight tying, as in GPT-2

def load_pretrained_gpt2(model_name):
    model = GPTModel(get_config(model_name))
    sd = download_gpt2_state_dict(model_name)
    load_weights_into_gpt(model, sd)
    del sd
    free_memory()
    return model.eval()

## 3. LoRA from scratch
For each linear layer inside the transformer blocks, the original weight W is frozen and a low-rank update is learned:
output = x·Wᵀ + b + (α / r) · x·A·B, with A of shape [in, r] and B of shape [r, out].
B starts at zero, so training starts exactly from the pretrained model.

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.scaling = alpha / rank

    def forward(self, x):
        return (x @ self.A @ self.B) * self.scaling


class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)


def replace_linear_with_lora(module, rank, alpha):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            setattr(module, name, LinearWithLoRA(child, rank, alpha))
        else:
            replace_linear_with_lora(child, rank, alpha)


def merge_lora(module):
    """Fold each LoRA update into its linear layer: W <- W + (scaling * A @ B)^T, giving a plain GPTModel again."""
    for name, child in module.named_children():
        if isinstance(child, LinearWithLoRA):
            with torch.no_grad():
                delta = (child.lora.A @ child.lora.B) * child.lora.scaling        # [in, out]
                child.linear.weight += delta.T.to(child.linear.weight.dtype)      # nn.Linear stores [out, in]
            setattr(module, name, child.linear)
        else:
            merge_lora(child)

## 4. Data: Alpaca-cleaned for training, the same 110 test instructions as before for evaluation

In [ ]:
def format_input(entry):
    instruction_text = ("Below is an instruction that describes a task. "
                        "Write a response that appropriately completes the request."
                        f"\n\n### Instruction:\n{entry['instruction']}")
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""
    return instruction_text + input_text

def load_alpaca():
    try:
        path = hf_hub_download(repo_id="yahma/alpaca-cleaned", filename="alpaca_data_cleaned.json", repo_type="dataset")
        name = "Alpaca-cleaned (yahma/alpaca-cleaned)"
    except Exception as e:
        print("Alpaca-cleaned not available, using the original Stanford Alpaca file:", e)
        path = download("https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json", "alpaca_data.json")
        name = "Stanford Alpaca (tatsu-lab/stanford_alpaca)"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f), name

alpaca, alpaca_name = load_alpaca()
random.seed(123)
random.shuffle(alpaca)
print(f"{alpaca_name}: {len(alpaca):,} examples")

# Same test set as the first instruction fine-tuning (examples 935-1044 of the book's instruction data)
book_path = download("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json",
                     "instruction-data.json")
with open(book_path, "r", encoding="utf-8") as f:
    book_data = json.load(f)
train_n, test_n = int(len(book_data) * 0.85), int(len(book_data) * 0.1)
test_data = book_data[train_n:train_n + test_n]
print(f"Test instructions (same as before): {len(test_data)}")


class InstructionDataset(Dataset):
    """Each item = (token ids of prompt + response + end-of-text, length of the prompt part)."""
    def __init__(self, data, max_length):
        self.items, skipped = [], 0
        for e in data:
            prompt_ids = tokenizer.encode(format_input(e) + "\n\n### Response:\n")
            response_ids = tokenizer.encode(e["output"])
            ids = prompt_ids + response_ids
            if len(ids) + 1 > max_length or len(response_ids) == 0:
                skipped += 1
                continue
            self.items.append((ids, len(prompt_ids)))
        print(f"  kept {len(self.items):,} examples, skipped {skipped:,} (too long or empty)")

    def __getitem__(self, i):
        return self.items[i]

    def __len__(self):
        return len(self.items)


def collate_fn(batch, pad_token_id=50256, ignore_index=-100):
    max_len = max(len(ids) + 1 for ids, _ in batch)
    inputs, targets = [], []
    for ids, prompt_len in batch:
        seq = ids + [pad_token_id]                                   # end-of-text after the response
        seq = seq + [pad_token_id] * (max_len - len(seq))
        x = torch.tensor(seq[:-1])
        y = torch.tensor(seq[1:])
        y[:prompt_len - 1] = ignore_index                            # don't train on predicting the prompt
        y[len(ids):] = ignore_index                                  # padding after the first end-of-text
        inputs.append(x)
        targets.append(y)
    return torch.stack(inputs), torch.stack(targets)


val_raw = alpaca[-500:]
train_raw = alpaca[:-500]
if NUM_TRAIN_EXAMPLES:
    train_raw = train_raw[:NUM_TRAIN_EXAMPLES]
print("Training set:")
train_ds = InstructionDataset(train_raw, MAX_LENGTH)
print("Validation set:")
val_ds = InstructionDataset(val_raw, MAX_LENGTH)

torch.manual_seed(123)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Check the masking on one example: only the response tokens (and one end-of-text) count in the loss
x, y = collate_fn([train_ds[0]])
print("\nTokens in the loss for one example:", int((y != -100).sum()), "of", y.shape[1])
print("Text the loss is computed on:", repr(tokenizer.decode([t for t in y[0].tolist() if t != -100])[:200]))

## 5. Build the model: GPT-2 large weights + LoRA

In [ ]:
torch.manual_seed(123)
model = load_pretrained_gpt2(BASE_MODEL)
for p in model.parameters():
    p.requires_grad = False
for block in model.trf_blocks:                     # LoRA on every linear layer inside the transformer blocks
    replace_linear_with_lora(block, LORA_RANK, LORA_ALPHA)
model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,} | trainable (LoRA): {trainable:,} ({100 * trainable / total:.2f}%)")
RESULTS["model"] = {"base_model": BASE_MODEL, "total_params": total, "trainable_lora_params": trainable,
                    "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA, "dataset": alpaca_name,
                    "train_examples": len(train_ds), "val_examples": len(val_ds)}

## 6. Training (mixed precision, gradient checkpointing, gradient accumulation, warm-up + cosine learning rate)

In [ ]:
def loss_on_batch(x, y):
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        logits = model(x.to(device))
    return nn.functional.cross_entropy(logits.float().flatten(0, 1), y.to(device).flatten(), ignore_index=-100)

def val_loss(num_batches=25):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(val_loader):
            if i >= num_batches:
                break
            total += loss_on_batch(x, y).item()
            n += 1
    model.train()
    return total / n

def answer(instruction, input_text="", max_new_tokens=150, mdl=None):
    mdl = mdl or model
    mdl.eval()
    prompt = format_input({"instruction": instruction, "input": input_text})
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        out = generate(mdl, text_to_token_ids(prompt, tokenizer).to(device), max_new_tokens=max_new_tokens,
                       context_size=1024, eos_id=50256)
    return token_ids_to_text(out, tokenizer)[len(prompt):].replace("### Response:", "").strip()

SAMPLE = "Give three tips for staying healthy."
print("Before fine-tuning:\n", answer(SAMPLE))

# Gradient checkpointing: each transformer block recomputes its activations during the backward pass
# instead of storing them, which cuts GPU memory a lot (at about 30% extra compute)
from torch.utils.checkpoint import checkpoint

def enable_gradient_checkpointing(model):
    for block in model.trf_blocks:
        original_forward = block.forward
        def forward(x, _f=original_forward):
            return checkpoint(_f, x, use_reentrant=False) if torch.is_grad_enabled() else _f(x)
        block.forward = forward

enable_gradient_checkpointing(model)
free_memory()

if os.path.exists(MODEL_FILE):
    print(f"\n{MODEL_FILE} already exists: skipping training (it is loaded later)")
    trained_here = False
else:
    trained_here = True
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.0)
    total_steps = math.ceil(len(train_loader) * NUM_EPOCHS / GRAD_ACCUM_STEPS)
    warmup = min(100, total_steps // 10)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: (s + 1) / warmup if s < warmup else
        0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup))))
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    print(f"\nOptimizer steps planned: {total_steps:,} (time limit {MAX_TRAIN_MINUTES} min)")
    history = {"step": [], "train_loss": [], "val_loss": []}
    start, step, running, stop = time.time(), 0, [], False
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for epoch in range(NUM_EPOCHS):
        for i, (x, y) in enumerate(train_loader):
            loss = loss_on_batch(x, y) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            running.append(loss.item() * GRAD_ACCUM_STEPS)
            if (i + 1) % GRAD_ACCUM_STEPS != 0:
                continue
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            step += 1

            if step % 200 == 0 or step == total_steps:
                tl, vl = sum(running) / len(running), val_loss()
                running = []
                history["step"].append(step); history["train_loss"].append(tl); history["val_loss"].append(vl)
                mins = (time.time() - start) / 60
                print(f"Ep {epoch + 1} step {step:5d}/{total_steps} | train loss {tl:.3f} | val loss {vl:.3f} | "
                      f"{mins:.1f} min, about {mins / step * (total_steps - step):.0f} min left")
            if step % 1000 == 0:
                print("  sample:", answer(SAMPLE, max_new_tokens=80)[:300].replace("\n", " "))
            if (time.time() - start) / 60 > MAX_TRAIN_MINUTES:
                print(f"Time limit of {MAX_TRAIN_MINUTES} minutes reached at step {step}: stopping training here")
                stop = True
                break
        if stop:
            break
    train_minutes = (time.time() - start) / 60
    print(f"Training finished: {step} optimizer steps in {train_minutes:.1f} minutes")

    plt.figure(figsize=(6, 3.5))
    plt.plot(history["step"], history["train_loss"], label="Training loss")
    plt.plot(history["step"], history["val_loss"], linestyle="-.", label="Validation loss")
    plt.xlabel("Optimizer step"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
    plt.savefig("alpaca-lora-loss.png", dpi=150)
    plt.show()
    RESULTS["training"] = {"optimizer_steps": step, "minutes": round(train_minutes, 1),
                           "final_train_loss": round(history["train_loss"][-1], 3) if history["train_loss"] else None,
                           "final_val_loss": round(history["val_loss"][-1], 3) if history["val_loss"] else None}
    del optimizer, scaler
    free_memory()

## 7. Merge LoRA into the weights and save a plain GPT model
After merging, the model is an ordinary `GPTModel` again (no LoRA code needed to use it), saved in half precision (~1.6 GB).

In [ ]:
if trained_here:
    x, _ = collate_fn([val_ds[0]])
    model.eval()
    with torch.no_grad():
        before = model(x.to(device)).float()
    merge_lora(model)
    with torch.no_grad():
        after = model(x.to(device)).float()
    print(f"Largest change in logits after merging: {(before - after).abs().max().item():.5f} (should be close to 0)")
    torch.save({k: v.half() for k, v in model.state_dict().items()}, MODEL_FILE)
    print("Saved", MODEL_FILE)
else:
    del model
    free_memory()
    model = GPTModel(get_config(BASE_MODEL))
    model.load_state_dict(torch.load(MODEL_FILE, map_location="cpu"))
    model.to(device)
    print("Loaded", MODEL_FILE)
model.eval()

for q, inp in [("Give three tips for staying healthy.", ""),
               ("Convert the active sentence to passive voice.", "The chef cooks the meal every day."),
               ("What is the capital of France?", ""),
               ("Rewrite the sentence in a formal tone.", "Hey, can you send me the report asap?"),
               ("Explain what photosynthesis is in two sentences.", "")]:
    print(f"### {q} {inp}\n{answer(q, inp, max_new_tokens=150)}\n")

## 8. Responses for the 110 test instructions (same test set as the first model)

In [ ]:
from tqdm.auto import tqdm
RESPONSES_FILE = "alpaca-lora-test-responses.json"
if os.path.exists(RESPONSES_FILE):
    with open(RESPONSES_FILE, "r") as f:
        test_data = json.load(f)
    print("Loaded saved responses")
else:
    for entry in tqdm(test_data, desc="Generating responses"):
        entry["model_response"] = answer(entry["instruction"], entry["input"], max_new_tokens=256)
    with open(RESPONSES_FILE, "w") as f:
        json.dump(test_data, f, indent=4)

for entry in test_data[:3]:
    print(format_input(entry))
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {entry['model_response']}")
    print("-" * 60)
RESULTS["example"] = {"instruction": format_input(test_data[0]), "reference": test_data[0]["output"],
                      "model_response": test_data[0]["model_response"]}
del model
free_memory()

## 9. Evaluation with Llama 3 as a judge (same method and prompt as before)

In [ ]:
import subprocess, shutil

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

if shutil.which("ollama") is None:
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434")
        break
    except Exception:
        time.sleep(2)
sh("ollama pull llama3")

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    data = {"model": model, "messages": [{"role": "user", "content": prompt}],
            "options": {"seed": 123, "temperature": 0, "num_ctx": 2048}}
    request = urllib.request.Request(url, data=json.dumps(data).encode("utf-8"), method="POST")
    request.add_header("Content-Type", "application/json")
    text = ""
    with urllib.request.urlopen(request) as response:
        for line in response:
            line = line.decode("utf-8").strip()
            if line:
                text += json.loads(line)["message"]["content"]
    return text

scores = []
for entry in tqdm(test_data, desc="Scoring entries"):
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model_response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              f"Respond with the integer number only.")
    reply = query_model(prompt)
    try:
        scores.append(int(reply.strip()))
    except ValueError:
        print("Could not convert score:", reply)

avg = sum(scores) / len(scores)
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {avg:.2f}")
RESULTS["llm_judge"] = {"judge": "llama3 (8B) via Ollama", "num_scored": len(scores),
                        "num_test": len(test_data), "average_score": round(avg, 2)}

## 10. Summary

In [ ]:
with open("results_alpaca_lora.json", "w") as f:
    json.dump(RESULTS, f, indent=2)
print(json.dumps(RESULTS, indent=2))